# librosa & torchaudio

**Domain:** Speech & Audio  ·  **recommended addition**  ·  **runnable:** yes

A refresher on the **two foundational Python audio libraries**. They compute the *same DSP*
(STFT → mel → log/MFCC) but live in different worlds: **librosa** is an eager NumPy/SciPy
analysis toolkit; **torchaudio** is the same transforms as batched, GPU-ready, differentiable
`nn.Module`s inside your PyTorch graph. Know which to reach for — and the loading gotcha that
bites everyone.

## 1. What & Why

Two libraries, same math, different jobs.

- **librosa** — a **NumPy/SciPy-stack toolkit for audio *analysis***. Load a clip, compute
  spectrograms / mel / MFCC / chroma, detect onsets, beats, tempo, and pitch. Everything is
  eager and returns plain `np.ndarray`. It is the de-facto standard for **music information
  retrieval (MIR)**, quick prototyping, and one-off feature engineering on CPU.

- **torchaudio** — **PyTorch's audio library**. Fast file I/O, the *same* transforms
  (`Spectrogram`, `MelSpectrogram`, `MFCC`, `Resample`, `FrequencyMasking`) expressed as
  composable `nn.Module`s that are **batched, GPU-accelerated, and differentiable**, plus
  ready-made datasets and **pretrained speech models** (Wav2Vec2, HuBERT, Tacotron2) under
  `torchaudio.pipelines`. Built to live *inside* a training / inference loop.

**Reach for librosa** when you are analyzing or prototyping on CPU and want the richest MIR
toolbox with the least ceremony. **Reach for torchaudio** when audio is the *input to a model*
and you need batching, GPU, on-the-fly differentiable augmentation, or a pretrained encoder.
They coexist: a common pattern is to *explore* features in librosa, then port the hot path to
torchaudio transforms so it runs batched on the GPU during training. Related notebooks:
[`ffmpeg`](ffmpeg.ipynb) (decode/convert before either), [`wav2vec`](wav2vec.ipynb) and
[`whisper-stt`](whisper-stt.ipynb) (models that consume these features).

## 2. Mental Model

**Same pipeline, two homes.** Every feature here walks one path:

```
 waveform ──► [ frame + window ] ──► [ FFT ] ──► |·|²  ──► [ mel filterbank ] ──► [ log / DCT ]
  (samples)      n_fft, hop                     power      n_mels                 dB  /  MFCC
```

- **librosa** = *"SciPy/MATLAB for audio."* You call a function, you get a NumPy matrix back at
  each step. Eager, CPU, single-array-at-a-time. Great for *looking at* audio.
- **torchaudio** = *"the same DSP as Lego `nn.Module`s in your compute graph."* You instantiate
  `T.MelSpectrogram()` **once**, move it to the GPU, and call it on `(batch, channels, time)`
  tensors with **gradients flowing through** — so a SpecAugment mask or a learnable front-end is
  just part of the model.

And the contract that trips everyone up — **loading defaults differ**:

| | `librosa.load(path)` | `torchaudio.load(path)` |
|---|---|---|
| sample rate | **resamples to 22050 Hz** | **keeps native rate** |
| channels | **downmixes to mono** | **keeps all channels** |
| return | `np.float32` shape `(T,)` | `torch.float32` shape `(channels, T)` |

Internalize that table and half your "why is the pitch wrong / shape off?" bugs disappear.

## 3. Key Concepts

- **Sample rate (`sr`) & Nyquist.** Samples per second; the highest representable frequency is
  `sr/2`. Everything downstream (STFT bin spacing, mel range) is in `sr` units, so a wrong `sr`
  silently corrupts pitch, tempo, and feature scale.
- **Waveform shape conventions.** librosa: `(T,)` mono or `(channels, T)` with `mono=False`.
  torchaudio: **always** `(channels, T)`. Mixing the two means `squeeze`/`unsqueeze`.
- **STFT params.** `n_fft` (window/FFT size → frequency resolution), `hop_length` (stride →
  time resolution), `win_length`. Frame count ≈ `1 + len(y)//hop_length` (with default
  `center=True` padding). Output has `n_fft//2 + 1` frequency bins.
- **Spectrogram.** `|STFT|` is *magnitude*; `|STFT|²` is *power*. librosa's
  `feature.melspectrogram` returns **power** by default.
- **Mel scale & filterbank.** A perceptual frequency warping; a mel filterbank (`n_mels`,
  `fmin`, `fmax`) collapses the linear-frequency spectrogram into mel bands.
- **MFCC.** A DCT of the log-mel spectrogram — a compact, decorrelated feature. *Defaults differ
  across the two libraries* (DCT type, normalization, log vs dB), so values **won't match**.
- **dB conversion.** librosa: `power_to_db` / `amplitude_to_db` (with `ref`, `top_db`).
  torchaudio: the `AmplitudeToDB` transform. Apply **once** — don't double-log.
- **Resampling.** librosa: `librosa.resample`. torchaudio: the `Resample` transform, which
  **caches its filter kernel** and runs batched on the GPU — instantiate it once, reuse it.
- **Transforms vs functional.** torchaudio has a stateful `transforms` API (`nn.Module`s you put
  in a model) and a stateless `functional` API (`F.resample`, `F.spectrogram`). librosa is all
  functions.

## 4. Setup

```bash
pip install librosa torchaudio soundfile
# torchaudio's version must match your torch version (install them together).
# Format support: librosa/soundfile use libsndfile (wav/flac/ogg out of the box);
# mp3 and exotic codecs need a system ffmpeg — see the ffmpeg notebook.
```

The worked examples below are deliberately **dependency-light**: Examples 1–3 run on **CPU with
only NumPy / SciPy / core PyTorch** (no `librosa`, no `torchaudio` import needed) so the notebook
executes top-to-bottom in any kernel. Example 4 shows the **real library API**, gated behind an
availability check so it still runs whether or not the packages are installed.

In [1]:
import importlib.util
import numpy as np

print("numpy", np.__version__)

# Report what's available; the runnable examples don't depend on these being present.
for mod in ("librosa", "torchaudio", "torch", "soundfile", "scipy"):
    have = importlib.util.find_spec(mod) is not None
    print(f"  {mod:<11} {'available' if have else 'not installed (only needed for Example 4)'}")

numpy 2.5.0
  librosa     not installed (only needed for Example 4)
  torchaudio  not installed (only needed for Example 4)
  torch       available
  soundfile   not installed (only needed for Example 4)
  scipy       available


## 5. Worked Examples

Examples 1–3 make the core ideas concrete with no audio download: **(1)** the shared
mel-spectrogram pipeline that *both* libraries implement, **(2)** the load-contract gotcha
(resample + shape), and **(3)** torchaudio's value-prop — a **batched, differentiable** STFT —
using only core `torch`. Example 4 is the real librosa + torchaudio API, gated so it always
runs.

### Example 1 — The mel pipeline both libraries compute

`librosa.feature.melspectrogram(...)` and `torchaudio.transforms.MelSpectrogram(...)` are
convenience wrappers over the exact steps below. Implementing them in ~20 lines of NumPy
demystifies the knobs (`n_fft`, `hop_length`, `n_mels`) and shows what shape you get back.

In [2]:
import numpy as np

sr = 16000                      # sample rate (Hz)
dur = 1.0
t = np.linspace(0, dur, int(sr * dur), endpoint=False)
# A two-tone test signal: 440 Hz (A4) + 880 Hz (A5).
y = 0.6 * np.sin(2 * np.pi * 440 * t) + 0.4 * np.sin(2 * np.pi * 880 * t)
y = y.astype(np.float32)

n_fft, hop, n_mels = 512, 256, 40

# 1) Frame + Hann window, then real FFT -> power spectrogram (n_fft//2+1, n_frames).
window = np.hanning(n_fft).astype(np.float32)
n_frames = 1 + (len(y) - n_fft) // hop
frames = np.stack([y[i*hop : i*hop + n_fft] * window for i in range(n_frames)], axis=1)
power = np.abs(np.fft.rfft(frames, axis=0)) ** 2          # (n_fft//2+1, n_frames)

# 2) A (toy) triangular mel filterbank: n_mels bands evenly over the mel axis.
def hz_to_mel(f):  return 2595 * np.log10(1 + f / 700)
def mel_to_hz(m):  return 700 * (10 ** (m / 2595) - 1)
mel_pts = mel_to_hz(np.linspace(hz_to_mel(0), hz_to_mel(sr / 2), n_mels + 2))
bins = np.floor((n_fft + 1) * mel_pts / sr).astype(int)
fb = np.zeros((n_mels, n_fft // 2 + 1), np.float32)
for m in range(1, n_mels + 1):
    l, c, r = bins[m-1], bins[m], bins[m+1]
    if c > l: fb[m-1, l:c] = (np.arange(l, c) - l) / (c - l)
    if r > c: fb[m-1, c:r] = (r - np.arange(c, r)) / (r - c)

# 3) Apply filterbank, convert to dB (power_to_db, ref=max).
mel_power = fb @ power                                     # (n_mels, n_frames)
mel_db = 10 * np.log10(np.maximum(mel_power, 1e-10))
mel_db -= mel_db.max()

print(f"signal: {y.shape[0]} samples @ {sr} Hz")
print(f"power spectrogram: {power.shape}  (freq_bins, frames)")
print(f"mel spectrogram:   {mel_db.shape}  (n_mels, frames), dB range "
      f"[{mel_db.min():.1f}, {mel_db.max():.1f}]")
# Loudest mel band should sit near our 440/880 Hz tones.
loud = mel_db.mean(axis=1).argmax()
print(f"loudest mel band ~ {mel_to_hz(np.linspace(0, hz_to_mel(sr/2), n_mels+2))[loud+1]:.0f} Hz")

signal: 16000 samples @ 16000 Hz
power spectrogram: (257, 61)  (freq_bins, frames)
mel spectrogram:   (40, 61)  (n_mels, frames), dB range [-138.6, 0.0]
loudest mel band ~ 445 Hz


That is the whole game: frame → window → FFT → power → mel → dB. librosa gives you each
intermediate as a NumPy array; torchaudio fuses them into one `MelSpectrogram` module. The knobs
(`n_fft`, `hop_length`, `n_mels`, `fmin/fmax`) mean the same thing in both.

### Example 2 — The load-contract gotcha (resample + shape)

The single most common bug: **`librosa.load` silently resamples to 22050 Hz and downmixes to
mono**, while **`torchaudio.load` keeps the native rate and channel layout**. If you assume the
file's original `sr`, every pitch/tempo/feature number is off. Here we simulate both contracts on
a stereo 48 kHz "file" so the difference is concrete — no audio file needed.

In [3]:
import numpy as np
from scipy.signal import resample_poly

native_sr = 48000
t = np.linspace(0, 0.5, int(native_sr * 0.5), endpoint=False)
left  = np.sin(2 * np.pi * 220 * t)
right = np.sin(2 * np.pi * 330 * t)
stereo = np.stack([left, right]).astype(np.float32)        # (channels, T) — torchaudio layout

print(f"on-disk 'file':              shape {stereo.shape}, sr {native_sr}")

# --- torchaudio.load contract: native sr, keep channels, (C, T) tensor ---
ta_wave, ta_sr = stereo, native_sr
print(f"torchaudio.load(path):       shape {ta_wave.shape}, sr {ta_sr}  (native + stereo)")

# --- librosa.load(path) default contract: sr=22050, mono=True, (T,) array ---
target_sr = 22050
mono = stereo.mean(axis=0)                                  # downmix
lib_wave = resample_poly(mono, target_sr, native_sr).astype(np.float32)
print(f"librosa.load(path):          shape {lib_wave.shape}, sr {target_sr}  (resampled + mono)")

# --- librosa.load(path, sr=None): keep native rate, still mono ---
lib_native = mono
print(f"librosa.load(path, sr=None): shape {lib_native.shape}, sr {native_sr}  (native, mono)")

print("\nLesson: pass sr=None to librosa to AVOID the silent 22050 Hz resample,")
print("and remember torchaudio gives you (channels, T) — squeeze for mono pipelines.")

on-disk 'file':              shape (2, 24000), sr 48000
torchaudio.load(path):       shape (2, 24000), sr 48000  (native + stereo)
librosa.load(path):          shape (11025,), sr 22050  (resampled + mono)
librosa.load(path, sr=None): shape (24000,), sr 48000  (native, mono)

Lesson: pass sr=None to librosa to AVOID the silent 22050 Hz resample,
and remember torchaudio gives you (channels, T) — squeeze for mono pipelines.


### Example 3 — torchaudio's value-prop: a batched, differentiable STFT

torchaudio's reason to exist is that transforms are `nn.Module`s — **batched, GPU-able, and
differentiable**. You don't need `torchaudio` itself to see this: core `torch.stft` already gives
a batched, autograd-tracked spectrogram. This is exactly what `T.Spectrogram` wraps.

In [4]:
import importlib.util

if importlib.util.find_spec("torch") is None:
    print("torch not installed — skipping (pip install torch). Example is correct as written.")
else:
    import torch

    sr, n_fft, hop = 16000, 512, 256
    tt = torch.linspace(0, 1, sr)
    # A BATCH of 4 clips at once — (batch, time). librosa can't do this in one call.
    freqs = torch.tensor([110., 220., 440., 880.])
    batch = torch.sin(2 * torch.pi * freqs[:, None] * tt)      # (4, 16000)
    batch.requires_grad_(True)                                 # track gradients

    window = torch.hann_window(n_fft)
    spec = torch.stft(batch, n_fft=n_fft, hop_length=hop, window=window,
                      center=True, return_complex=True)        # (4, 257, frames)
    mag = spec.abs()
    print(f"batched spectrogram: {tuple(mag.shape)}  (batch, freq_bins, frames)")

    # Differentiable: a scalar 'loss' on the spectrogram backprops to the waveform.
    loss = mag.mean()
    loss.backward()
    print(f"gradient flows to waveform: grad shape {tuple(batch.grad.shape)}, "
          f"||grad|| = {batch.grad.norm():.4f}")
    print("=> SpecAugment, learnable front-ends, and GPU batching all 'just work'.")

batched spectrogram: (4, 257, 63)  (batch, freq_bins, frames)
gradient flows to waveform: grad shape (4, 16000), ||grad|| = 0.0454
=> SpecAugment, learnable front-ends, and GPU batching all 'just work'.


Batched and differentiable is the whole point: the spectrogram is a node in your graph, so a
data-augmentation mask or a trainable filterbank is part of the model, and the same code runs on
the GPU by `.to('cuda')`. That is what librosa — pure eager NumPy — cannot give you.

### Example 4 — The real API (gated behind availability)

The actual library calls. Each branch is skipped if the package isn't installed, but the code is
correct as written — note how the **same** mel-spectrogram intent reads in each library, and how
torchaudio's transform is an object you instantiate once and reuse.

In [5]:
import importlib.util
import numpy as np

sr = 16000
y = (0.6 * np.sin(2 * np.pi * 440 * np.linspace(0, 1, sr))).astype(np.float32)

# ---- librosa ----
if importlib.util.find_spec("librosa"):
    import librosa
    # y, sr = librosa.load("clip.wav", sr=None)   # sr=None keeps native rate!
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=512, hop_length=256, n_mels=40)
    S_db = librosa.power_to_db(S, ref=np.max)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    print(f"librosa    mel {S_db.shape}  mfcc {mfcc.shape}")
else:
    print("librosa not installed -> librosa.feature.melspectrogram(y=y, sr=sr, n_mels=40)")

# ---- torchaudio ----
if importlib.util.find_spec("torchaudio") and importlib.util.find_spec("torch"):
    import torch
    import torchaudio.transforms as T
    wav = torch.from_numpy(y)[None]                 # (1, T); torchaudio uses (channels, T)
    melspec = T.MelSpectrogram(sample_rate=sr, n_fft=512, hop_length=256, n_mels=40)
    to_db = T.AmplitudeToDB(stype="power")
    S2 = to_db(melspec(wav))                        # instantiate once, reuse / .to('cuda')
    print(f"torchaudio mel {tuple(S2.shape)}  (batch, n_mels, frames)")
    # resampler = T.Resample(orig_freq=sr, new_freq=8000)  # cached kernel, GPU-able
else:
    print("torchaudio not installed -> T.MelSpectrogram(sample_rate=sr, n_mels=40)(wav)")

librosa not installed -> librosa.feature.melspectrogram(y=y, sr=sr, n_mels=40)
torchaudio not installed -> T.MelSpectrogram(sample_rate=sr, n_mels=40)(wav)


## 6. Gotchas & Pitfalls

- **`librosa.load` silently resamples to 22050 Hz and downmixes to mono.** Pass `sr=None` to
  keep the native rate; pass `mono=False` to keep channels. A wrong `sr` corrupts every pitch /
  tempo / mel computation downstream. (See Example 2.)
- **Shape conventions clash.** librosa is `(T,)` for mono; torchaudio is **always** `(channels,
  T)`. Bridge with `torch.from_numpy(y)[None]` and `tensor.squeeze().numpy()`.
- **Power vs amplitude vs dB — don't double-log.** `librosa.feature.melspectrogram` returns
  **power**; convert with `power_to_db`, not `amplitude_to_db`. In torchaudio, set
  `AmplitudeToDB(stype="power")` to match. Applying dB twice gives nonsense.
- **MFCC values won't match across libraries.** Different DCT type, normalization, and log-vs-dB
  defaults. Pick one library for a given pipeline; don't compare numbers across them.
- **Frame counts differ subtly.** Both default to `center=True` (pad by `n_fft//2`), but edge
  handling and rounding can give off-by-one frame counts. Don't hard-code frame indices.
- **librosa is single-threaded NumPy — keep it out of the training hot loop.** For per-batch
  feature extraction use torchaudio transforms (GPU, batched) or precompute features once.
- **Cache the `Resample` transform.** It builds a sinc filter kernel on construction;
  re-instantiating it every call (or using `F.resample` in a loop) is needlessly slow.
- **Integer PCM must be normalized.** Expect `float32` in `[-1, 1]`. An `int16` array loaded raw
  is in `[-32768, 32767]` — divide by 32768 or your spectrogram scale is meaningless.
- **torchaudio backend / codecs.** I/O goes through a backend (`soundfile` or `ffmpeg`); MP3 and
  some formats need a system **ffmpeg**. Install one or decode upstream with the
  [`ffmpeg`](ffmpeg.ipynb) notebook's tooling.
- **Version lock-step.** `torchaudio` must match your `torch` version, or imports fail at load.

## 7. When to Use vs Alternatives

| Tool | Best for | Trade-off |
|---|---|---|
| **librosa** | MIR, analysis, prototyping features on CPU; richest toolbox (beat, chroma, pitch, onset) | Eager NumPy, single sample at a time, no GPU/autograd — slow in a training loop |
| **torchaudio** | Audio as model input: batched/GPU/differentiable transforms, datasets, pretrained Wav2Vec2/HuBERT | Tied to PyTorch; fewer high-level MIR features than librosa; version must match torch |
| **`scipy.signal`** | Low-level filtering, resampling, raw STFT without an audio-specific API | No mel/MFCC/MIR conveniences; you assemble the pipeline yourself |
| **`soundfile` / `pydub`** | Pure read/write & simple edits (trim, concat, format convert) | Not feature extraction — pair with one of the above |
| **`ffmpeg`** ([notebook](ffmpeg.ipynb)) | Decoding, format/codec conversion, the universal front door | A CLI/binary, not a feature library — use it *before* librosa/torchaudio |
| **`nnAudio` / `torchlibrosa`** | GPU, differentiable spectrograms with librosa-compatible numbers | Niche; torchaudio now covers most of this |
| **`essentia` / Kaldi** | Heavy-duty MIR (essentia) or classic ASR feature stacks (Kaldi) | Steeper setup; reach for them only when librosa/torchaudio fall short |

**Rule of thumb:** *analyzing* audio or doing MIR on CPU → **librosa**. Audio feeding a
*PyTorch model* (batched, GPU, augmentation, pretrained encoders) → **torchaudio**. Most real
projects use **both**: librosa to explore, torchaudio in the training pipeline. Need to *decode/
convert* first? That's **ffmpeg**, upstream of either.

## 8. Resources

- **librosa documentation** — API, feature reference, and example gallery:
  https://librosa.org/doc/latest/index.html
- **torchaudio documentation** — transforms, functional, pipelines, datasets:
  https://pytorch.org/audio/stable/index.html
- **torchaudio tutorials** — Audio I/O, feature extraction, Wav2Vec2 ASR, augmentation:
  https://pytorch.org/audio/stable/tutorials/
- **McFee et al., "librosa: Audio and Music Signal Analysis in Python" (SciPy 2015)** — the
  design paper behind the library:
  https://librosa.org/doc/latest/index.html#citing-librosa
- **librosa feature extraction guide** — mel, MFCC, chroma, tempo, onsets in one place:
  https://librosa.org/doc/latest/feature.html
- **torchaudio `pipelines` (pretrained bundles)** — Wav2Vec2/HuBERT encoders ready to use:
  https://pytorch.org/audio/stable/pipelines.html